# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all available record sets by @id and name
print("Available record sets (by @id and name):")
record_sets = []
for rs in metadata.record_sets:
    print(f"  @id: {rs.id}  |  name: {rs.name}")
    record_sets.append(rs.id)

# For each record set, list fields and columns
for rs in metadata.record_sets:
    print(f"\nRecord set '@id': {rs.id} -- name: {rs.name}\n  Fields:")
    for field in rs.fields:
        columns = []
        if hasattr(field, 'columns') and field.columns:
            columns = [col.id for col in field.columns]
        print(f"    Field @id: {field.id}  |  name: {field.name}  |  dataType: {field.data_type}  |  columns: {columns}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Example: extract data from each record set and load into DataFrames, indexed by their @id
dataframes = {}

# We'll print shape and columns for each record set
for rs_id in record_sets:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Record set @id: {rs_id}, DataFrame shape: {df.shape}")
        print(f"Columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"Could not load records for {rs_id}: {e}")

# Choose one record set for subsequent analysis
selected_rs_id = record_sets[0] if record_sets else None
if selected_rs_id and selected_rs_id in dataframes:
    print(dataframes[selected_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# --- EDA on the selected record set ---
df = dataframes[selected_rs_id] if selected_rs_id else None

# Identify numeric columns by checking types in first row (dictionary)
numeric_columns = []
if df is not None and not df.empty:
    for col in df.columns:
        # Try parsing values as float
        try:
            df[col].astype(float)
            numeric_columns.append(col)
        except:
            continue

print(f"Numeric columns available: {numeric_columns}")

# We'll use the first numeric field for demo purposes
if numeric_columns:
    numeric_field_id = numeric_columns[0]
    print(f"Using numeric field @id for filtering: {numeric_field_id}")
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    # Remove NaN
    valid_df = df[df[numeric_field_id].notna()]
    threshold = valid_df[numeric_field_id].mean() if valid_df[numeric_field_id].count()>0 else 0
    filtered_df = valid_df[valid_df[numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > mean ({threshold:.3f}): {len(filtered_df)} rows\n")
    print(filtered_df[[numeric_field_id]].head())
    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / (filtered_df[numeric_field_id].std() if filtered_df[numeric_field_id].std() else 1)
    print(f"\nNormalized records for {numeric_field_id}:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    # Try to group by the first non-numeric field
    non_numeric_columns = [col for col in df.columns if col not in numeric_columns]
    group_field = non_numeric_columns[0] if non_numeric_columns else None
    if group_field:
        print(f"\nGrouping by field @id: {group_field}")
        try:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(grouped_df[[numeric_field_id]].head())
        except Exception as e:
            print(f"Unable to group, error: {e}")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

if numeric_columns:
    plt.figure(figsize=(8,4))
    df[numeric_field_id].hist(bins=30, alpha=0.7)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    # If grouping was available, make a boxplot
    if group_field:
        plt.figure(figsize=(12,5))
        df.boxplot(column=numeric_field_id, by=group_field)
        plt.title(f'{numeric_field_id} by {group_field}')
        plt.suptitle("")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- This notebook demonstrated how to load and explore a Croissant-formatted dataset using the `mlcroissant` library.
- We loaded dataset and record set metadata, dynamically extracted DataFrames by record set `@id`, performed some basic EDA, and visualized a numeric field's distribution.
- The structure and schema of the dataset (record sets, fields, columns, and their `@id`s) can be programmatically discovered and used for robust and reproducible analysis pipelines.